<a href="https://colab.research.google.com/github/sinbbang/BigData/blob/main/12%EC%9E%A5_%EC%A3%BC%EA%B0%84%EB%B3%B4%EA%B3%A0%EC%84%9C_Colab%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12장 · 엑셀 합치기부터 Word 보고서까지

**수강생용 Colab 실습 노트북**

Colab의 왼쪽 파일 탭에 문의결과_1주.xlsx, 문의결과_2주.xlsx, 문의결과_3주.xlsx, 문의결과_4주.xlsx, 담당자배정표.xlsx를 업로드하세요. 주간보고서_완성예시.pdf는 결과 비교용입니다.

처음에는 셀 1~4까지만 차례로 실행합니다. 3주차의 열 이름을 고치고 셀 4에서 통과를 확인한 뒤 5~10번을 실행하세요. 실습에는 API 키가 필요하지 않습니다.

## 셀 1 · 준비 도구

실행 후 준비 완료와 차트에 쓸 글꼴 이름이 나오는지 확인하세요. 처음 설치할 때는 시간이 걸릴 수 있습니다.

In [1]:
# 표 읽기, 차트 그리기, Word 문서 작성에 필요한 도구를 준비합니다.
import importlib.util  # 도구가 이미 설치되어 있는지 확인할 때 씁니다.
import subprocess  # 필요한 도구를 설치하는 명령을 파이썬에서 실행할 때 씁니다.
import sys  # 현재 노트북을 실행하는 파이썬의 위치를 알아낼 때 씁니다.
from pathlib import Path  # 파일이 있는지 확인하거나 파일 이름의 일부를 꺼낼 때 씁니다.

# 왼쪽은 코드에서 불러오는 이름, 오른쪽은 설치할 때 쓰는 이름입니다.
# 예를 들어 Word 도구는 python-docx로 설치하고 docx로 불러옵니다.
패키지 = {"pandas": "pandas", "openpyxl": "openpyxl", "matplotlib": "matplotlib", "docx": "python-docx"}
# 도구를 하나씩 확인해, 찾을 수 없는 도구의 설치 이름만 모읍니다.
없는_패키지 = [이름 for 모듈, 이름 in 패키지.items() if importlib.util.find_spec(모듈) is None]
if 없는_패키지:  # 설치할 도구가 하나라도 있으면 아래 줄을 실행합니다.
    # 현재 파이썬에서 pip로 빠진 도구만 설치합니다. -q는 설치 중 화면에 나오는 글을 줄입니다.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *없는_패키지])

import pandas as pd  # 엑셀 자료를 표로 다룰 때 pandas를 pd라고 줄여 씁니다.
import matplotlib.pyplot as plt  # 차트를 그릴 때 pyplot을 plt라고 줄여 씁니다.
from matplotlib import font_manager  # 차트에 사용할 수 있는 글꼴 목록을 읽는 도구입니다.
from docx import Document  # 새 Word 문서를 만들거나 저장된 문서를 다시 열 때 씁니다.
from docx.oxml.ns import qn  # Word에서 한글 글꼴을 따로 지정할 때 씁니다.
# 문서의 길이는 인치(Inches), 글자 크기는 포인트(Pt)로 지정합니다.
from docx.shared import Inches, Pt

# 차트의 제목, 축 이름, 범례에 한글이 보이도록 글꼴을 정합니다.
# 사용하고 싶은 글꼴을 우선순위 순으로 적습니다.
후보 = ["NanumGothic", "Noto Sans CJK KR", "Malgun Gothic", "Apple SD Gothic Neo", "Noto Sans CJK JP"]
# 지금 차트를 그릴 때 쓸 수 있는 글꼴의 이름을 모읍니다.
설치된_글꼴 = {글꼴.name for 글꼴 in font_manager.fontManager.ttflist}
# 후보 중 설치된 첫 글꼴을 고릅니다. 하나도 없으면 None이 됩니다.
한글_글꼴 = next((글꼴 for 글꼴 in 후보 if 글꼴 in 설치된_글꼴), None)
if 한글_글꼴:  # 후보 글꼴을 찾았으면 그 글꼴을 사용합니다.
    plt.rcParams["font.family"] = [한글_글꼴]  # 이 뒤에 그리는 차트의 기본 글꼴을 정합니다.
else:  # 후보 글꼴을 찾지 못하면 아래에서 한글 표시를 준비합니다.
    # 한글 글꼴을 설정해 주는 도구가 있는지 확인합니다.
    if importlib.util.find_spec("koreanize_matplotlib") is None:
        # 도구가 없을 때만 설치합니다.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "koreanize-matplotlib"])
    import koreanize_matplotlib  # 이 도구를 불러오면 차트에 쓸 한글 글꼴이 설정됩니다.
# 차트의 음수 눈금에 일반 빼기 기호(-)를 쓰도록 합니다.
plt.rcParams["axes.unicode_minus"] = False
# 준비가 끝났다는 메시지와 실제로 설정된 글꼴 이름을 보여 줍니다.
print("준비 완료. 차트 글꼴:", plt.rcParams["font.family"][0])

준비 완료. 차트 글꼴: NanumGothic


## 셀 2 · 실습 설정

첫 시도에는 빈 중괄호 그대로 실행합니다. 셀 4에서 원인을 찾은 뒤 슬라이드 23의 설정을 적용하세요.

In [2]:
# 읽을 파일과 보고서에 표시할 정보를 정합니다.
# 파일을 1주부터 순서대로 적습니다. 셀 7에서는 마지막 두 파일의 주차를 비교합니다.
파일목록 = ["문의결과_1주.xlsx", "문의결과_2주.xlsx", "문의결과_3주.xlsx", "문의결과_4주.xlsx"]
배정표_파일 = "담당자배정표.xlsx"  # 각 문의 유형을 누가 맡는지 적어 둔 엑셀 파일입니다.
# 처음에는 {}로 둡니다. 검사 뒤 {"원래 열 이름": "바꿀 열 이름"} 형식으로 적습니다.
열이름_맞추기 = {}
보고서_제목 = "늘봄마켓 고객 문의 주간 보고"  # Word 문서 첫 줄에 표시할 제목입니다.
# 보고서에 표시할 기간입니다. 실제로 읽을 주차는 위의 파일목록에서 정합니다.
기간 = "2026년 11월 1~4주"
작성부서 = "고객지원팀"  # 보고서 제목 아래에 표시할 부서 이름입니다.
# 여섯 단계의 완료 표시를 모두 False(아직 끝나지 않음)로 되돌립니다.
검사_통과 = 집계_완료 = 차트_완료 = 문장_완료 = 문서_완료 = 보고서_검증 = False
# 설정을 마친 뒤 실행할 셀 번호를 보여 줍니다.
print("설정을 읽었습니다. 다음은 셀 3입니다.")

설정을 읽었습니다. 다음은 셀 3입니다.


## 셀 3 · 주차별 파일 합치기

파일마다 행 수와 열 이름을 비교하세요. 문의 번호와 접수일은 원본에 적힌 값을 씁니다.

In [4]:
# 네 주의 엑셀 표를 위아래로 붙이고, 각 행에 몇 주차 문의인지 표시합니다.
# 파일을 다시 읽으므로 이전 검사와 보고서 작성 단계의 완료 표시를 초기화합니다.
검사_통과 = 집계_완료 = 차트_완료 = 문장_완료 = 문서_완료 = 보고서_검증 = False
# 주차별 파일과 배정표가 모두 있는지 확인해, 찾을 수 없는 파일 이름만 모읍니다.
누락파일 = [이름 for 이름 in [*파일목록, 배정표_파일] if not Path(이름).is_file()]
if 누락파일:  # 빠진 파일이 하나라도 있는지 봅니다.
    # 빠진 파일 이름을 쉼표로 이어서 보여 주고 실행을 멈춥니다.
    raise FileNotFoundError("Colab 왼쪽 파일 탭에 업로드한 뒤 다시 실행하세요: " + ", ".join(누락파일))

조각들 = []  # 나중에 합칠 주차별 표를 담을 빈 목록입니다.
주차순서 = []  # '1주', '2주'처럼 파일을 읽은 순서대로 주차를 기록할 목록입니다.
원본행수 = {}  # 파일 이름과 원래 행 수를 짝지어 기록할 빈 사전입니다.
for 파일 in 파일목록:  # 목록에서 파일을 하나씩 꺼내 아래 작업을 반복합니다.
    # 확장자를 떼고 마지막 밑줄 뒤를 읽습니다. '문의결과_3주.xlsx'라면 '3주'입니다.
    주차 = Path(파일).stem.rsplit("_", 1)[-1]
    if 주차 in 주차순서:  # 지금 읽는 주차가 앞에서 이미 나왔는지 확인합니다.
        # 같은 주차를 두 번 더하지 않도록 해당 주차를 알려 주고 멈춥니다.
        raise ValueError(f"같은 주차의 파일이 두 번 지정되었습니다: {주차}")
    # 해당 엑셀 파일의 '분류결과' 시트를 표로 읽습니다.
    표 = pd.read_excel(파일, sheet_name="분류결과", engine="openpyxl")
    # 이 파일을 읽었을 때의 행 수를 적어 둡니다. 셀 4에서 합친 행 수와 비교합니다.
    원본행수[파일] = len(표)
    # 셀 2에 적은 규칙대로 열 이름을 바꿉니다. 규칙이 비어 있으면 원래 이름을 사용합니다.
    표 = 표.rename(columns=열이름_맞추기)
    # 이름을 바꾼 뒤 같은 이름의 열이 둘 이상인지 검사합니다.
    if 표.columns.duplicated().any():
        # 중복된 열이 생겼다면 어느 파일인지 알려 주고 멈춥니다.
        raise ValueError(f"{파일}에서 이름이 같은 열이 두 개 생겼습니다. 셀 2를 확인하세요.")
    표["주차"] = 주차  # 이 표의 모든 행에 같은 주차를 적습니다.
    조각들.append(표)  # 다 읽은 표를 합치기 전 목록에 넣습니다.
    # 이번 주차를 목록 끝에 더합니다. 표와 차트에서도 이 순서를 씁니다.
    주차순서.append(주차)
    # 파일마다 행 수와 열 이름을 보여 줍니다. 다른 주차와 이름이 다른 열을 찾아보세요.
    print(f"{파일}: {len(표)}행", list(표.columns))
# 같은 이름의 열끼리 맞춰서 표를 위아래로 붙이고, 행 번호를 0부터 다시 매깁니다.
# 어떤 파일에 해당 열이 없으면 그 주차의 행은 그 열에서 빈칸이 됩니다.
전체 = pd.concat(조각들, ignore_index=True)
print("합친 표:", len(전체), "행")  # 합친 표의 전체 행 수를 보여 줍니다.

문의결과_1주.xlsx: 30행 ['id', '접수일', '채널', '내용', 'category', 'sentiment', 'summary', 'needs_review', '주차']
문의결과_2주.xlsx: 32행 ['id', '접수일', '채널', '내용', 'category', 'sentiment', 'summary', 'needs_review', '주차']
문의결과_3주.xlsx: 35행 ['id', '접수일', '채널', '내용', '유형', 'sentiment', 'summary', 'needs_review', '주차']
문의결과_4주.xlsx: 38행 ['id', '접수일', '채널', '내용', 'category', 'sentiment', 'summary', 'needs_review', '주차']
합친 표: 135 행


## 셀 4 · 행 수·열 이름·빈칸 검사

여기서 잠시 멈추세요. 행 수가 맞는데도 빈칸이 있다면 몇 건이며 어느 주차인지 적으세요. 셀 2를 수정한 뒤에는 2, 3, 4번을 차례로 다시 실행합니다.

In [5]:
# 파일을 합치며 행이 빠졌는지, 열 이름이 맞는지, 유형이 빈 문의가 있는지 확인합니다.
# 합친 표에 꼭 필요한 열입니다. category는 문의 유형, needs_review는 검토 필요 여부입니다.
기대열 = ["id", "접수일", "채널", "내용", "category", "sentiment", "summary", "needs_review", "주차"]
# 파일별로 기록한 행 수를 모두 더합니다. 이 값이 합친 표의 행 수와 같아야 합니다.
원본_행수 = sum(원본행수.values())
# 실제 표에 있지만 기대한 목록에는 없는 열을 찾습니다.
남는열 = [열 for 열 in 전체.columns if 열 not in 기대열]
# 기대한 목록에는 있지만 실제 표에는 없는 열을 찾습니다.
빠진열 = [열 for 열 in 기대열 if 열 not in 전체.columns]
if "category" in 전체.columns:  # 문의 유형을 담는 category 열이 있는지 확인합니다.
    # 빈값을 빈 문자열로 바꾸고 앞뒤 공백을 지웁니다. 값이 비면 그 행에 True를 표시합니다.
    빈유형 = 전체["category"].fillna("").astype(str).str.strip().eq("")
else:  # 유형 열 자체가 없는 경우입니다.
    # 모든 행을 '유형이 비어 있음'(True)으로 표시합니다.
    빈유형 = pd.Series(True, index=전체.index)
빈칸 = int(빈유형.sum())  # True를 하나에 1건씩 세어 유형이 빈 문의 수를 구합니다.
# 원본 네 파일의 행 수 합계와 합친 표의 행 수를 나란히 보여 줍니다.
print(f"① 행 수: 원본 합계 {원본_행수} / 합친 결과 {len(전체)}")
# 예상 밖의 열과 빠진 열을 각각 보여 줍니다. 목록이 비면 '없음'을 표시합니다.
print("② 열 목록: 예상 밖의 열", 남는열 or "없음", "/ 빠진 열", 빠진열 or "없음")
print("③ category 빈칸:", 빈칸, "건")  # 유형이 비어 있는 문의가 몇 건인지 보여 줍니다.
if 빈칸:  # 빈칸이 한 건이라도 있을 때만 해당 주차를 찾습니다.
    # 유형이 빈 행만 골라 주차를 읽습니다. 같은 주차가 여럿이면 한 번만 보여 줍니다.
    print("빈칸이 있는 주차:", 전체.loc[빈유형, "주차"].unique().tolist())
# 행 수가 맞고, 남거나 빠진 열이 없고, 유형 빈칸도 0건이어야 통과입니다.
검사_통과 = (원본_행수 == len(전체)) and not 남는열 and not 빠진열 and 빈칸 == 0
if 검사_통과:  # 앞의 세 검사를 통과하면 문의 번호도 확인합니다.
    # id에 값이 없거나 같은 번호가 두 번 이상 나오는지 봅니다.
    if 전체["id"].isna().any() or 전체["id"].duplicated().any():
        검사_통과 = False  # 번호에 문제가 있으면 통과 표시를 False로 바꿉니다.
        # 어떤 항목을 고쳐야 하는지 화면에 알려 줍니다.
        print("문의 번호에 빈칸 또는 중복이 있습니다.")
# 통과 여부에 따라 셀 5로 갈지, 셀 2~4를 다시 실행할지 안내합니다.
print("통과. 다음은 셀 5입니다." if 검사_통과 else "확인 필요. 셀 2의 열 이름 규칙을 고친 뒤 셀 2~4를 다시 실행하세요.")

① 행 수: 원본 합계 135 / 합친 결과 135
② 열 목록: 예상 밖의 열 ['유형'] / 빠진 열 없음
③ category 빈칸: 35 건
빈칸이 있는 주차: ['3주']
확인 필요. 셀 2의 열 이름 규칙을 고친 뒤 셀 2~4를 다시 실행하세요.


## 셀 5 · 집계표와 요약 엑셀

검사 통과 후 실행하세요. 왼쪽 파일 탭에서 주간보고_요약.xlsx를 열어 세 시트를 확인합니다.

In [6]:
# 문의 유형별로 각 주의 건수를 세고, 결과를 한 엑셀 파일의 세 시트에 저장합니다.
if not 검사_통과:  # 셀 4의 검사를 통과했는지 먼저 확인합니다.
    # 검사를 통과하지 못했으면 이번 집계를 시작하기 전에 멈춥니다.
    raise ValueError("셀 4의 검사를 먼저 통과하세요. 이번 실행의 요약 파일을 만들지 않았습니다.")
# 행에는 유형, 열에는 주차를 놓고 각 칸의 문의 수를 셉니다.
# 파일을 읽은 순서대로 주차 열을 놓고, 빠진 주차 열은 0으로 채웁니다.
집계 = pd.crosstab(전체["category"], 전체["주차"]).reindex(columns=주차순서, fill_value=0)
# axis=1은 한 행을 가로로 더한다는 뜻입니다. 유형별로 네 주의 합계를 구합니다.
집계["합계"] = 집계.sum(axis=1)
# 합계가 큰 유형부터 정렬합니다. 합계가 같으면 정렬 전의 순서를 유지합니다.
집계 = 집계.sort_values("합계", ascending=False, kind="stable")
# 유형별 합계를 다시 더하면 원본의 전체 문의 수와 같은지 확인합니다.
if int(집계["합계"].sum()) != 원본_행수:
    # 두 값이 다르면 엑셀을 저장하기 전에 멈춥니다.
    raise ValueError("집계표와 원본의 총건수가 다릅니다.")
# 사람 확인 표시(needs_review)의 모든 값이 True 또는 False인지 확인합니다.
if not 전체["needs_review"].isin([True, False]).all():
    # 다른 값이 있으면 확인 목록을 만들기 전에 멈춥니다.
    raise ValueError("needs_review에 True 또는 False 이외의 값이 있습니다.")
# 사람이 확인해야 하는 행(True)만 고른 뒤 주차, 문의 번호, 유형, 요약 네 열을 복사합니다.
확인필요 = 전체.loc[전체["needs_review"].eq(True), ["주차", "id", "category", "summary"]].copy()
# with 안에서 같은 엑셀 파일에 세 시트를 씁니다. 이 블록을 마치면 파일이 저장됩니다.
with pd.ExcelWriter("주간보고_요약.xlsx", engine="openpyxl") as 엑셀:
    # 집계표를 첫 시트에 씁니다. 표의 왼쪽에 있는 문의 유형도 함께 저장합니다.
    집계.to_excel(엑셀, sheet_name="유형별_주차별")
    # 검토할 문의를 두 번째 시트에 씁니다. index=False는 별도의 행 번호 열을 만들지 않습니다.
    확인필요.to_excel(엑셀, sheet_name="확인필요", index=False)
    # 네 주를 합친 전체 자료를 세 번째 시트에 씁니다.
    전체.to_excel(엑셀, sheet_name="합친데이터", index=False)
집계_완료 = True  # 집계표와 요약 엑셀을 만드는 단계가 끝났다고 표시합니다.
print(집계.to_string())  # 집계표를 글자로 바꾸어 화면에 보여 줍니다.
# 유형별 합계와 전체 문의 행 수를 나란히 보여 줍니다.
print("집계표 총합:", int(집계["합계"].sum()), "/ 합친 행 수:", len(전체))

ValueError: 셀 4의 검사를 먼저 통과하세요. 이번 실행의 요약 파일을 만들지 않았습니다.

## 셀 6 · 주차별 차트

배송 선의 4주차 값과 집계표를 비교하세요. 미분류 1건은 집계에 보관하고 차트에는 다섯 업무 유형을 그립니다.

In [ ]:
# 유형마다 선 하나를 그려 주차별 문의 수를 비교하고, Word에 넣을 그림을 저장합니다.
if not 집계_완료:  # 셀 5에서 집계표를 만들었는지 확인합니다.
    # 집계표가 준비되지 않았으면 셀 5를 먼저 실행하도록 안내하고 멈춥니다.
    raise ValueError("셀 5에서 이번 실행의 집계를 완성하세요.")
# 그림 한 장과 선을 그릴 영역(축)을 만듭니다. 그림 크기는 가로 7인치, 세로 3.8인치입니다.
그림, 축 = plt.subplots(figsize=(7, 3.8))
for 유형 in 집계.index:  # 집계표 왼쪽에 있는 문의 유형을 하나씩 꺼냅니다.
    if 유형 != "미분류":  # '미분류'는 집계표에 남겨 두고 차트에는 업무 유형만 그립니다.
        # 가로축은 주차, 세로축은 그 유형의 문의 수입니다. 동그라미로 각 주의 값을 표시합니다.
        # 실습에서 배송 선의 마지막 점은 집계표의 ‘배송’ 행과 ‘4주’ 열이 만나는 칸의 값입니다.
        축.plot(주차순서, 집계.loc[유형, 주차순서].astype(int), marker="o", label=유형)
# 차트 제목과 두 축의 이름을 붙입니다.
축.set(title="주차별 유형별 문의 건수", xlabel="주차", ylabel="건수")
축.set_ylim(bottom=0)  # 세로축을 0건에서 시작하게 합니다.
축.legend(fontsize=8)  # 어느 선이 어떤 문의 유형인지 보여 주는 범례를 넣습니다.
그림.tight_layout()  # 제목과 축 이름이 잘리지 않도록 그림 안의 여백을 조정합니다.
# Word에 넣을 차트를 chart.png로 저장합니다. dpi는 그림의 해상도입니다.
그림.savefig("chart.png", dpi=150)
plt.show()  # Colab의 셀 아래에 차트를 보여 줍니다.
plt.close(그림)  # 화면에 보여 준 그림을 닫아 다음 차트를 그릴 준비를 합니다.
차트_완료 = True  # Word에 넣을 차트를 준비했다고 표시합니다.

## 셀 7 · 보고서의 핵심 문장

금주 증감과 전체 기간의 배송 비율을 구별해서 읽으세요.

In [ ]:
# 네 주 전체의 합계와 지난주 대비 변화를 구해 보고서에 넣을 다섯 문장을 만듭니다.
if not 집계_완료:  # 보고서 문장을 만드는 데 필요한 집계표가 준비됐는지 확인합니다.
    # 집계를 마치지 않았으면 셀 5를 먼저 실행하도록 안내하고 멈춥니다.
    raise ValueError("셀 5의 집계표를 먼저 만드세요.")
총건수 = len(전체)  # 문의 한 건이 한 행이므로 전체 행 수가 곧 문의 건수입니다.
# 집계표를 합계가 큰 순으로 정렬했으므로 첫 행의 이름이 가장 많은 유형입니다.
최다유형 = 집계.index[0]
# 그 유형의 행과 '합계' 열이 만나는 칸에서 네 주의 건수를 읽습니다.
최다건수 = int(집계.loc[최다유형, "합계"])
# 네 주간 그 유형의 건수를 전체 건수로 나눕니다. 예를 들어 135건 중 47건이면 약 34.8%입니다.
최다비율 = 최다건수 / 총건수 * 100
# 목록의 마지막 두 주를 순서대로 꺼냅니다. 실습 파일에서는 지난주가 3주, 이번 주가 4주입니다.
지난주, 이번주 = 주차순서[-2:]
# 이번 주 열을 세로로 더해 모든 유형의 이번 주 문의 수를 구합니다.
이번주_건수 = int(집계[이번주].sum())
지난주_건수 = int(집계[지난주].sum())  # 지난주 열도 같은 방법으로 더합니다.
# 네 주 전체에서 가장 많았던 유형의 이번 주 건수를 읽습니다.
최다_이번주 = int(집계.loc[최다유형, 이번주])
# 같은 유형의 지난주 건수를 읽어 이번 주와 비교할 준비를 합니다.
최다_지난주 = int(집계.loc[최다유형, 지난주])

# '이번 주 건수 - 지난주 건수'를 받아 보고서에 쓸 문장으로 바꾸는 함수입니다.
def 지난주와_비교(차이):
    if 차이 > 0:  # 차이가 양수이면 문의가 늘어난 경우입니다.
        return f"지난주보다 {차이}건 늘었습니다."  # 늘어난 건수를 넣은 문장을 돌려줍니다.
    if 차이 < 0:  # 차이가 음수이면 문의가 줄어든 경우입니다.
        # abs는 부호를 뺀 크기입니다. 차이가 -3이면 '3건 줄었습니다'라고 씁니다.
        return f"지난주보다 {abs(차이)}건 줄었습니다."
    return "지난주와 같습니다."  # 늘거나 줄지 않았으면 두 주의 건수가 같다고 씁니다.

핵심문장 = [  # 다섯 문장을 보고서에 넣을 순서대로 목록에 담습니다.
    # f로 시작하면 중괄호 자리에 실제 값이 들어갑니다. 여기서는 기간과 총건수를 넣습니다.
    f"{기간} 고객 문의는 총 {총건수}건입니다.",
    # 가장 많은 유형, 건수, 비율을 적습니다. .1f는 소수점 아래 한 자리까지 표시합니다.
    f"가장 많은 유형은 {최다유형}({최다건수}건, {최다비율:.1f}%)입니다.",
    # 이번 주 전체 건수와 지난주보다 몇 건 늘거나 줄었는지 적습니다.
    f"이번 주({이번주}) 문의는 {이번주_건수}건으로 {지난주와_비교(이번주_건수 - 지난주_건수)}",
    # 전체 기간에 가장 많았던 유형 한 가지에 대해 같은 두 주를 비교합니다.
    f"{최다유형} 문의는 이번 주 {최다_이번주}건으로 {지난주와_비교(최다_이번주 - 최다_지난주)}",
    # 검토할 문의의 행 수를 세어 적고, 자세한 목록이 있는 보고서 4절을 안내합니다.
    f"사람 확인이 필요한 문의는 {len(확인필요)}건입니다. 4절 목록을 확인하세요.",
]
print("\n".join(핵심문장))  # 문장 사이에 줄바꿈을 넣어 화면에 한 줄씩 보여 줍니다.
문장_완료 = True  # Word에 넣을 다섯 문장이 준비됐다고 표시합니다.

## 셀 8 · 팀의 Word 보고서

주간보고서.docx를 내려받아 제목, 핵심 숫자, 집계표, 차트, 검토 목록을 확인합니다. 숫자는 완성 예시 PDF와 대조할 수 있습니다.

In [ ]:
# 팀 보고서에 제목, 핵심 숫자, 집계표, 차트, 검토할 문의를 순서대로 넣습니다.
# Word에 넣을 차트와 핵심 문장이 모두 준비됐는지 확인합니다.
if not (차트_완료 and 문장_완료):
    # 하나라도 빠졌으면 문서를 만들기 전에 멈춥니다.
    raise ValueError("셀 6과 셀 7을 먼저 실행하세요.")

def 새_문서():  # 팀 보고서와 담당자별 문서에 공통으로 쓸 빈 Word 문서를 만드는 함수입니다.
    문서 = Document()  # 빈 Word 문서를 하나 만듭니다.
    # 첫 번째 구역을 선택합니다. 이 구역에서 용지 크기와 여백을 정합니다.
    구역 = 문서.sections[0]
    # 세로 11.69인치, 가로 8.27인치로 A4 세로 방향에 맞춥니다.
    구역.page_height, 구역.page_width = Inches(11.69), Inches(8.27)
    # 용지 위쪽과 아래쪽 여백을 각각 0.72인치로 맞춥니다.
    구역.top_margin = 구역.bottom_margin = Inches(0.72)
    # 용지 좌우 여백을 각각 0.8인치로 맞춥니다.
    구역.left_margin = 구역.right_margin = Inches(0.8)
    # 본문, 문서 제목, 절 제목, 글머리표 문단의 서식을 차례로 설정합니다.
    for 이름 in ["Normal", "Title", "Heading 1", "List Bullet"]:
        스타일 = 문서.styles[이름]  # 방금 고른 이름에 해당하는 Word 스타일을 가져옵니다.
        # 해당 스타일의 기본 글꼴을 맑은 고딕으로 지정합니다.
        스타일.font.name = "Malgun Gothic"
        # Word가 한글을 표시할 때도 '맑은 고딕'을 쓰게 지정합니다.
        스타일.element.get_or_add_rPr().rFonts.set(qn("w:eastAsia"), "맑은 고딕")
    기본 = 문서.styles["Normal"]  # 본문에 해당하는 Normal 스타일을 선택합니다.
    기본.font.size = Pt(10)  # 본문 글자를 10포인트로 정합니다.
    # 본문 한 문단이 끝날 때 아래에 3포인트의 간격을 둡니다.
    기본.paragraph_format.space_after = Pt(3)
    # 각 절의 제목은 본문보다 큰 16포인트로 정합니다.
    문서.styles["Heading 1"].font.size = Pt(16)
    return 문서  # 설정을 마친 문서를 돌려줍니다.

# 열 제목과 여러 행의 값을 받아 Word 문서에 표를 넣는 함수입니다.
def 표_추가(문서, 제목들, 행들):
    # 제목이 들어갈 첫 행을 만듭니다. 열 수는 제목의 개수와 같습니다.
    표 = 문서.add_table(rows=1, cols=len(제목들))
    표.style = "Table Grid"  # 표의 칸 경계가 보이도록 테두리 서식을 적용합니다.
    # zip으로 첫 행의 칸과 열 제목을 왼쪽부터 하나씩 짝짓습니다.
    for 칸, 제목 in zip(표.rows[0].cells, 제목들):
        # 그 칸의 첫 문단에 제목을 글자로 넣고 굵게 표시합니다.
        칸.paragraphs[0].add_run(str(제목)).bold = True
    for 행 in 행들:  # 표에 넣을 데이터를 한 행씩 읽습니다.
        # Word 표에 새 행을 하나 추가한 뒤, 그 행의 칸과 데이터 값을 순서대로 짝짓습니다.
        for 칸, 값 in zip(표.add_row().cells, 행):
            칸.text = str(값)  # 숫자나 날짜도 칸에 쓸 수 있도록 글자로 바꿔 넣습니다.
    return 표  # 완성한 표를 돌려줍니다.

문서 = 새_문서()  # 위에서 정한 용지와 글꼴 설정으로 팀 보고서를 시작합니다.
# 셀 2에서 정한 보고서 제목을 맨 위에 넣습니다. 0은 문서 전체의 제목을 뜻합니다.
문서.add_heading(보고서_제목, 0)
# 제목 아래 한 줄에 보고 기간과 작성 부서를 표시합니다.
문서.add_paragraph(f"기간: {기간}    작성: {작성부서}")
# 첫 절의 제목을 넣습니다. 1은 절 제목의 수준을 뜻합니다.
문서.add_heading("1. 핵심 숫자", 1)
for 문장 in 핵심문장:  # 셀 7에서 만든 다섯 문장을 차례로 꺼냅니다.
    # 각 문장을 글머리표가 붙은 한 문단으로 넣습니다.
    문서.add_paragraph(문장, style="List Bullet")
# 둘째 절에 유형별·주차별 집계표가 나올 자리를 표시합니다.
문서.add_heading("2. 유형별·주차별 건수", 1)
# 첫 열의 제목은 '유형'입니다. *집계.columns는 그 뒤에 주차별 열 이름과 '합계'를 펼쳐 넣습니다.
표_추가(문서, ["유형", *집계.columns],
       # 유형 이름, 주차별 건수, 합계를 한 행에 넣습니다. int(v)는 건수를 정수로 바꿉니다.
       ([유형, *[int(v) for v in 행]] for 유형, 행 in 집계.iterrows()))
문서.add_page_break()  # 다음 내용이 새 페이지에서 시작되도록 쪽을 나눕니다.
# 새 페이지에 차트가 들어갈 셋째 절의 제목을 넣습니다.
문서.add_heading("3. 주차별 추이", 1)
# 셀 6에서 저장한 chart.png를 가로 5.8인치로 넣습니다.
문서.add_picture("chart.png", width=Inches(5.8))
# 검토할 문의를 모아 보여 줄 넷째 절의 제목을 넣습니다.
문서.add_heading("4. 사람 확인이 필요한 문의", 1)
# 확인필요 표에서 주차, 문의 번호, 유형, 요약을 한 행씩 꺼내 Word 표에 넣습니다.
표_추가(문서, ["주차", "id", "유형", "요약"], 확인필요.itertuples(index=False, name=None))
# 독자가 보고서 작성 방법과 배포 전 검토 절차를 알 수 있게 안내 문장을 넣습니다.
문서.add_paragraph("※ 이 보고서는 프로그램이 원본 엑셀 파일로 자동 작성했습니다. 담당자 검토 후 배포합니다.")
문서.save("주간보고서.docx")  # 내용을 주간보고서.docx라는 이름으로 저장합니다.
문서_완료 = True  # 셀 9에서 저장된 문서를 검사할 수 있도록 완료 표시를 켭니다.
print("주간보고서.docx를 저장했습니다.")  # 저장한 파일 이름을 화면에 보여 줍니다.

## 셀 9 · 저장 파일 다시 열어 검증

원본·집계·문서의 총건수와 검토 목록의 문의 번호를 대조합니다. 슬라이드 27의 여섯 항목도 사람이 확인하세요.

In [ ]:
# 저장한 엑셀과 Word를 다시 열어 총건수와 검토할 문의 번호를 원본과 맞춰 봅니다.
if not 문서_완료:  # 이번 실행에서 팀 보고서를 저장했는지 확인합니다.
    # 보고서가 준비되지 않았으면 셀 8을 먼저 실행하도록 안내하고 멈춥니다.
    raise ValueError("셀 8의 Word 문서가 준비되지 않았습니다.")
import re  # 문장 속에서 정해진 모양의 숫자를 찾는 도구입니다.
저장문서 = Document("주간보고서.docx")  # 방금 저장한 Word 파일을 다시 엽니다.
# Word의 각 문단에서 글자를 꺼내 줄바꿈으로 잇습니다. 표의 내용은 아래에서 따로 읽습니다.
저장문장 = "\n".join(문단.text for 문단 in 저장문서.paragraphs)
# '고객 문의는 총'과 '건' 사이의 숫자를 찾습니다. '1,350'처럼 쉼표가 있어도 됩니다.
찾은값 = re.search(r"고객 문의는 총\s*([\d,]+)건", 저장문장)
# 찾은 숫자에서 쉼표를 빼고 정수로 바꿉니다. 문장을 찾지 못하면 None으로 둡니다.
문서건수 = int(찾은값.group(1).replace(",", "")) if 찾은값 else None
# 엑셀 파일도 다시 열어 '유형별_주차별' 시트를 읽습니다.
저장집계 = pd.read_excel("주간보고_요약.xlsx", sheet_name="유형별_주차별", engine="openpyxl")
# 저장된 엑셀에서 유형별 '합계' 열을 더해 총문의 수를 구합니다.
집계총합 = int(저장집계["합계"].sum())
# Word 첫 번째 표에서 제목 행을 건너뛰고 각 행의 마지막 칸('합계')을 더합니다.
Word_집계합 = sum(int(행.cells[-1].text) for 행 in 저장문서.tables[0].rows[1:])
# Word 두 번째 표는 검토 목록입니다. 제목 행을 건너뛰고 두 번째 칸의 문의 번호를 모읍니다.
문서번호 = [행.cells[1].text for 행 in 저장문서.tables[1].rows[1:]]
# 원본에서 사람 확인이 필요한 행만 골라 문의 번호를 모읍니다.
원본번호 = [str(번호) for 번호 in 전체.loc[전체["needs_review"].eq(True), "id"]]
# 원본, 저장한 엑셀 표, Word 본문 문장, Word 집계표의 총건수가 모두 같은지 비교합니다.
보고서_검증 = (원본_행수 == 집계총합 == 문서건수 == Word_집계합
          # 번호를 정렬해 비교하면 목록 순서가 달라도 빠지거나 잘못 들어간 문의를 찾을 수
          # 있습니다.
          and sorted(문서번호) == sorted(원본번호))
print("① 원본 파일 행 수 합계:", 원본_행수)  # 원본 네 파일의 행 수 합계를 보여 줍니다.
# 저장한 엑셀에서 다시 계산한 총건수를 보여 줍니다.
print("② 저장된 엑셀 집계표 총합:", 집계총합)
# Word 본문에서 읽은 건수와 Word 표에서 더한 건수를 나란히 보여 줍니다.
print("③ Word의 문장:", 문서건수, "/ Word 표의 총합:", Word_집계합)
# Word 검토 목록의 행 수와 원본에서 확인 표시가 있는 문의 수를 보여 줍니다.
print("④ Word 검토 목록:", len(문서번호), "/ 원본 확인 표시:", len(원본번호))
if not 보고서_검증:  # 총건수나 검토할 문의 번호가 일치하지 않는지 봅니다.
    # 값이 다르면 이유를 알리고 실행을 멈춥니다.
    raise ValueError("저장된 파일과 원본의 숫자 또는 문의 번호가 서로 다릅니다.")
# 자동 검사를 통과하면 Word를 열어 숫자와 배치도 직접 확인하도록 안내합니다.
print("통과. 이제 Word를 직접 열어 숫자와 모양을 확인하세요.")

## 셀 10 · 담당자별 문서

이번 주 38건만 담당자 다섯 명에게 나눕니다. 미배정이 있으면 새 개인별 문서 생성을 멈춥니다. 이전 성공 실행의 파일은 남아 있을 수 있습니다. 파일 탭에서 완성한 결과물을 내려받으세요.

In [ ]:
# 이번 주 문의에 담당자를 붙이고, 각 담당자가 맡은 문의로 Word 파일을 하나씩 만듭니다.
if not 보고서_검증:  # 셀 9의 원본·엑셀·Word 비교를 통과했는지 확인합니다.
    # 검증을 마치지 않았으면 담당자별 자료를 만들기 전에 멈춥니다.
    raise ValueError("셀 9에서 이번 실행의 보고서 검사를 먼저 통과하세요.")
# 문의 유형과 담당자 이름이 짝지어진 배정표를 엽니다.
배정 = pd.read_excel(배정표_파일, engine="openpyxl")
# 전체 자료에서 이번 주 행만 골라 별도 표로 복사합니다.
이번주원본 = 전체.loc[전체["주차"].eq(이번주)].copy()
# category가 같은 행끼리 연결합니다. left는 담당자를 못 찾은 문의도 남겨 두라는 뜻입니다.
# 문의에는 같은 유형이 여러 번 나와도 됩니다. many_to_one은 배정표의 유형 중복을 검사합니다.
이번주표 = 이번주원본.merge(배정[["category", "담당자"]], on="category", how="left", validate="many_to_one")
# 담당자 이름이 없거나 공백만 있으면 해당 행에 True(미배정)를 표시합니다.
미배정 = 이번주표["담당자"].fillna("").astype(str).str.strip().eq("")
# ~는 True와 False를 뒤집습니다. 담당자가 있는 행만 True로 바꿔 그 수를 셉니다.
배정건수 = int((~미배정).sum())
# 담당자를 찾지 못한 문의 수를 보여 줍니다.
print("담당자 없는 문의:", int(미배정.sum()), "건")
# 담당자가 정해진 건수와 이번 주 전체 문의 수를 나란히 보여 줍니다.
print("배정된 건수:", 배정건수, "/ 이번 주 전체:", len(이번주원본))
if 미배정.any():  # 담당자가 없는 행이 한 건이라도 있는지 확인합니다.
    # 배정이 빠진 유형 이름을 중복 없이 골라 쉼표로 연결합니다.
    빠진유형 = ", ".join(이번주표.loc[미배정, "category"].unique())
    # 어떤 유형의 배정표를 고쳐야 하는지 알리고, 개인별 문서를 만들기 전에 멈춥니다.
    raise ValueError(f"담당자 없는 유형: {빠진유형}. 배정표를 고친 뒤 셀 10을 다시 실행하세요. 새 담당자 문서를 만들지 않았습니다.")

만든_건수 = 0  # 문서에 실제로 넣은 문의 수를 0건부터 누적합니다.
# 담당자 이름순으로, 이름과 그 사람이 맡은 문의 묶음을 하나씩 꺼냅니다.
for 담당자, 묶음 in 이번주표.groupby("담당자", sort=True):
    # 한 사람의 문의를 접수일 순으로, 같은 날짜 안에서는 문의 번호순으로 정렬합니다.
    묶음 = 묶음.sort_values(["접수일", "id"])
    담당문서 = 새_문서()  # 셀 8의 함수를 다시 써서 같은 용지와 글꼴로 새 문서를 만듭니다.
    # 담당자 이름과 해당 주차가 보이도록 문서 제목을 넣습니다.
    담당문서.add_heading(f"{담당자}님 담당 문의 ({기간} 중 {이번주})", 1)
    # 이 사람이 맡은 전체 건수와 그중 사람 확인이 필요한 건수를 적습니다.
    담당문서.add_paragraph(f"이번 주 담당 문의는 {len(묶음)}건입니다. 이 중 확인 필요 {int(묶음['needs_review'].sum())}건.")
    # 번호, 접수일, 요약을 꺼내고, 검토가 필요한 문의만 마지막 칸에 '예'를 넣습니다.
    행들 = ([행.id, 행.접수일, 행.summary, "예" if 행.needs_review else ""] for 행 in 묶음.itertuples())
    # 네 열의 제목과 방금 만든 행들로 담당자용 Word 표를 채웁니다.
    표_추가(담당문서, ["id", "접수일", "요약", "확인필요"], 행들)
    # 파일 이름에 담당자 이름을 넣어 사람마다 다른 파일에 저장하게 합니다.
    파일명 = f"담당자별_{담당자}.docx"
    담당문서.save(파일명)  # 현재 담당자의 Word 파일을 저장합니다.
    만든_건수 += len(묶음)  # 이번 파일에 담은 문의 수를 누적 건수에 더합니다.
    # 파일 이름과 그 안에 넣은 문의 수를 화면에 보여 줍니다.
    print(f"{파일명}: {len(묶음)}건")
# 모든 담당자의 문서에 담은 건수를 합해 이번 주 원본 문의 수와 비교합니다.
if 만든_건수 != len(이번주원본):
    # 두 값이 다르면 배정 결과를 다시 확인하도록 오류를 냅니다.
    raise ValueError("담당자별 합계가 이번 주 문의 수와 다릅니다.")
# 모두 맞으면 개인별 파일에 넣은 총건수와 이번 주 원본 건수를 보여 줍니다.
print("검증 통과. 담당자별 합계:", 만든_건수, "/ 이번 주 전체:", len(이번주원본))